# Day 3 — Multi-Agent Airport Operations Copilot

## Objective

Convert the Day 1 RAG pipeline and Day 2 operational tools
into an agentic AI system.

### Agent Architecture

User Query
→ Orchestrator Agent
→ Operations Investigator
→ Policy & Compliance Agent
→ Resolution Agent
→ Recommendation

### Concepts Covered

- Multi-agent architecture
- Agent orchestration
- Tool integration
- RAG integration
- ReAct-style workflow
- Agent handoffs
- Short-term conversational memory
- Iteration control
- Natural-language query handling

In [46]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/nineleaps/Documents/airport-ai-copilot


In [47]:
import chromadb

from src.embeddings import load_embedding_model
from src.memory import ConversationMemory
from src.agents import OrchestratorAgent

CHROMA_PATH = PROJECT_ROOT / ".chroma"

client = chromadb.PersistentClient(
    path=str(CHROMA_PATH)
)

collection = client.get_collection(
    name="airport_policies"
)

embedding_model = load_embedding_model()

memory = ConversationMemory(
    max_turns=5
)

print("Agents and RAG components loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7046.61it/s]


Agents and RAG components loaded.


In [48]:
orchestrator = OrchestratorAgent(
    collection=collection,
    embedding_model=embedding_model,
    max_iterations=5,
    memory=memory
)

print(orchestrator.name)

Orchestrator Agent


In [49]:
query = """
What is happening operationally at SFO
and what does the surge policy allow?
"""

result = orchestrator.run_query(query)

print("Status:", result["status"])
print("Airport:", result["airport_code"])

print("\nInvestigation")
print("----------------")
print("Severity:", result["investigation"]["severity"])

for finding in result["investigation"]["findings"]:
    print("-", finding)

print("\nRecommendations")
print("----------------")
for recommendation in result["resolution"]["recommendations"]:
    print("-", recommendation)

Status: success
Airport: SFO

Investigation
----------------
Severity: high
- Driver cancellation rate is elevated.
- Airport queue size is high.
- Average ETA is elevated.

Recommendations
----------------
- Consider increasing driver availability through targeted incentives.
- Investigate driver cancellation causes and consider targeted incentives.
- Increase available driver capacity to reduce passenger wait times.


## ReAct-Style Workflow

The orchestrator follows a controlled agent workflow:

PLAN
↓
INVESTIGATE
↓
CHECK POLICY
↓
GENERATE RESOLUTION
↓
COMPLETE

The implementation uses observable actions and results rather
than exposing the model's private chain-of-thought.

Maximum iterations are controlled using `max_iterations = 5`.

In [50]:
print("Agent Workflow")
print("=" * 40)

for step in result["steps"]:
    print(
        f"{step['iteration']}. "
        f"{step['action']}"
    )

Agent Workflow
1. PLAN
2. INVESTIGATE
3. CHECK_POLICY
4. GENERATE_RESOLUTION
5. COMPLETE


## Day 3 Architecture Summary

### Operations Investigator
Responsible for:
- Reading airport telemetry
- Identifying operational anomalies
- Determining severity
- Producing operational findings

### Policy & Compliance Agent
Responsible for:
- Searching the policy knowledge base
- Retrieving applicable policies
- Providing policy-grounded information

### Resolution Agent
Responsible for:
- Evaluating operational findings
- Generating possible interventions
- Producing recommendations

### Orchestrator Agent
Responsible for:
- Coordinating the agents
- Controlling the workflow
- Managing iterations
- Connecting investigation, policy and resolution

### Memory
Short-term conversation memory allows the system
to retain recent context across interactions.